# NB11 — Sensitivity analyses and paper assets

**Purpose.** Check that the main conclusions survive: (a) random within-hospital split instead of 2014→2015; (b) stricter/looser hospital eligibility; (c) logistic regression or XGBoost as the base model, or a feature set without GCS / sparse labs; (d) ECE binning choices. Then collect every table and figure needed for the manuscript into one zip.

**Research question.** Robustness

| | |
|---|---|
| **Inputs** | all previous outputs |
| **Outputs** | Table S4–S6, `results/paper_assets.zip` |
| **Free-Colab runtime** | 15–25 min |

**How to run:** `Runtime → Run all`. If Colab disconnects, just run all again — finished steps are cached.
At the end, download the zip printed in the last cell into your local `Plan A/v4/results/` folder.

In [ ]:
# ==== Setup (identical in every notebook) ====
import os, sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
BASE = os.environ['TRANSPORT_BASE']
sys.path.insert(0, os.path.join(BASE, 'code'))
import importlib, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
importlib.reload(tu)
warnings.filterwarnings('ignore')
tu.set_style()
OUT = tu.init_notebook('NB11')

In [ ]:
import json, joblib, lightgbm as lgb
from scipy.stats import wilcoxon
m, e = tu.load_analysis()
pred = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eicu.parquet'))
bp = json.load(open(os.path.join(tu.MODELS, 'lgbm_best_params.json')))
FEATS = tu.feature_set('V1')
SPLIT_COL = 'split'
upd = pred[pred[SPLIT_COL] == 'update'].reset_index(drop=True)
ev = pred[pred[SPLIT_COL] == 'eval'].reset_index(drop=True)
print(f'update n={len(upd):,} ({upd.hospital_id.nunique()} hospitals) | eval n={len(ev):,} ({ev.hospital_id.nunique()} hospitals)')

In [ ]:
KEY = ['S0_none', 'S1b_global_logistic', 'S2a_local_intercept', 'S2b_local_logistic', 'S3_eb']
def strategy_summary(upd, ev, pcol, label, min_n=tu.MIN_HOSP_N, min_events=tu.MIN_HOSP_EVENTS):
    evs, info, _ = tu.run_strategies(upd, ev, pcol)
    c = evs.groupby('hospital_id').agg(n=('mortality', 'size'), d=('mortality', 'sum'))
    nu = upd.groupby('hospital_id').size()
    elig = [h for h in c[(c.n >= min_n) & (c.d >= min_events)].index if nu.get(h, 0) >= tu.MIN_UPDATE_N]
    out = []
    for s in KEY:
        ph = tu.per_hospital(evs[evs['hospital_id'].isin(elig)], s, min_n=0, min_events=0, full=False)
        ma = tu.meta_analysis(ph['ln_oe'], ph['ln_oe_se'])
        out.append(dict(analysis=label, strategy=tu.STRATEGY_LABELS[s], hospitals=len(elig),
                        pct_acceptable=100 * ph['oe'].between(*tu.OE_ACCEPT).mean(),
                        median_abs_ln_oe=ph['ln_oe'].abs().median(), tau_ln_oe=ma['tau'],
                        pooled_oe=np.exp(ma['mu']), pi_lo=np.exp(ma['pi_lo']), pi_hi=np.exp(ma['pi_hi'])))
    return out

rows = []
rows += strategy_summary(upd, ev, 'p_lgbm', 'Primary: temporal 2014->2015, LightGBM, >=100 pts/>=10 deaths')
ru, re_ = pred[pred['random_split'] == 'update'], pred[pred['random_split'] == 'eval']
rows += strategy_summary(ru, re_, 'p_lgbm', 'Random within-hospital 50/50 split')
rows += strategy_summary(upd, ev, 'p_lgbm', 'Eligibility >=50 pts/>=5 deaths', 50, 5)
rows += strategy_summary(upd, ev, 'p_lgbm', 'Eligibility >=200 pts/>=20 deaths', 200, 20)
rows += strategy_summary(upd, ev, 'p_lr', 'Base model: logistic regression')
rows += strategy_summary(upd, ev, 'p_xgb', 'Base model: XGBoost')

# Feature-set sensitivity: GCS is scored differently in the two systems (verbal=1 for intubated patients in MIMIC,
# 25% missing and higher in eICU), and lactate/albumin/bilirubin/INR are measured far less often in eICU.
dev = m[m['period'] == 'dev']
def refit_pred(drop_prefixes):
    F = [f for f in FEATS if not any(f == d or f.startswith(d + '_') for d in drop_prefixes)]
    mdl = lgb.LGBMClassifier(n_estimators=bp['n_estimators'], subsample_freq=1, random_state=tu.SEED, verbose=-1,
                             n_jobs=-1, **bp['params']).fit(dev[F].astype(float), dev['mortality'].astype(int))
    pe = pd.Series(mdl.predict_proba(e.set_index('stay_id')[F].astype(float))[:, 1], index=e['stay_id'].values)
    return F, pe
for lab, drop in [('Feature set without GCS', ['gcs']),
                  ('Feature set without sparse labs (lactate, albumin, bilirubin, INR)', ['lactate', 'albumin', 'bili', 'inr'])]:
    F, pe = refit_pred(drop)
    pp = pred.copy(); pp['p_alt'] = pe.reindex(pp['stay_id']).values
    rows += strategy_summary(pp[pp[SPLIT_COL] == 'update'].reset_index(drop=True),
                             pp[pp[SPLIT_COL] == 'eval'].reset_index(drop=True), 'p_alt', lab + f' ({len(F)} features)')
sens = pd.DataFrame(rows)
tu.save_table(sens.round(3), 'tableS4_sensitivity')

In [ ]:
# ECE binning sensitivity (continuity with the old paper; not a primary metric)
def ece_quantile(y, p, n_bins=10):
    q = np.quantile(p, np.linspace(0, 1, n_bins + 1)); idx = np.clip(np.searchsorted(q, p, side='right') - 1, 0, n_bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(n_bins) if (idx == b).any())
evs = pd.read_parquet(os.path.join(tu.MODELS, 'pred_eval_strategies.parquet'))
ec = []
for s in ['S0_none', 'S1b_global_logistic', 'S3_eb']:
    y, p = evs['mortality'].values, evs[s].values
    ec.append(dict(strategy=tu.STRATEGY_LABELS[s], ece10=tu.ece(y, p, 10), ece15=tu.ece(y, p, 15), ece20=tu.ece(y, p, 20),
                   ece10_equal_mass=ece_quantile(y, p, 10), ici=tu.ici_metrics(y, p)['ici']))
ec = pd.DataFrame(ec)
tu.save_table(ec.round(5), 'tableS5_ece_binning')

### Collect paper assets

In [ ]:
import glob, shutil
assets = os.path.join(tu.RESULTS, 'paper_assets'); os.makedirs(assets, exist_ok=True)
manifest = []
for nb in ['NB03', 'NB04', 'NB05', 'NB06', 'NB07', 'NB08', 'NB09', 'NB10', 'NB11']:
    for f in sorted(glob.glob(os.path.join(tu.RESULTS, nb, '*'))):
        base_ = os.path.basename(f)
        if base_.startswith('_cache') or base_.endswith('.pkl'):
            continue
        if base_.startswith(('table', 'fig', 'cohort_flow', 'funnel', 'harm', 'alert', 'dca', 'head_to_head',
                             'meta_regression', 'domain', 'summary')):
            dst = os.path.join(assets, f'{nb}_{base_}'); shutil.copy(f, dst)
            manifest.append(dict(notebook=nb, file=base_))
manifest = pd.DataFrame(manifest)
manifest.to_csv(os.path.join(assets, 'MANIFEST.csv'), index=False)
tripod = pd.DataFrame([
    ('Title/abstract: identify as development + external validation/updating study', 'Manuscript'),
    ('Data sources, dates, setting', 'NB01-NB03, Table 1'),
    ('Participants: eligibility, flow', 'NB03 cohort_flow'),
    ('Outcome definition and timing', 'RESEARCH_PLAN s4; NB03'),
    ('Predictors: definition, timing, harmonisation', 'code/sql_queries.py; NB03 feature audit'),
    ('Sample size', 'NB03 flow; NB08 eligible hospitals'),
    ('Missing data handling', 'NB04 (native NaN for trees, train-median imputation for LR)'),
    ('Model development and tuning', 'NB04'),
    ('Performance measures: discrimination, calibration hierarchy, clinical utility', 'NB05, NB06, NB08'),
    ('Model updating', 'NB08, NB09'),
    ('Heterogeneity across clusters', 'NB06'),
    ('Fairness / subgroups', 'NB10'),
    ('Sensitivity analyses', 'NB11'),
    ('Code availability', 'code/ + notebooks/'),
], columns=['TRIPOD+AI item (abridged)', 'Where addressed'])
tripod.to_csv(os.path.join(assets, 'TRIPOD_AI_map.csv'), index=False)
zpath = shutil.make_archive(os.path.join(tu.RESULTS, 'paper_assets'), 'zip', assets)
print('paper assets:', len(manifest), 'files ->', zpath)

In [ ]:
tu.finish_notebook({'sensitivity': sens.round(4).to_dict('records'), 'ece_binning': ec.round(5).to_dict('records'),
                    'paper_assets_zip': zpath, 'n_assets': len(manifest)})